# NB6 · REINFORCE 与 baseline：策略梯度定理的实验闭环

这是实验室的最后一本，也是难度最高的一本（对应 L9/L10，★★★，约 50 分钟）。前几本里你手算过 Bellman、迭代过值函数、复现过 Q-learning——全是**价值路线**。这一本换赛道：**策略梯度**，直接对策略本身做梯度上升。

**前置**：主站 L9 的推导链「策略梯度定理」（20 步，第七轮清单 #10）与 L10 的「baseline 不改期望只降方差」（#11）。本本的三个自检格，正是这两条推导的实验版。

主线三步：

1. **手写完整 REINFORCE**——numpy softmax 策略 + 那行魔法导数 ∇lnπ = onehot(a) − π；
2. **无 baseline vs 有 baseline** 各训 300 回合；
3. **验证推导 #11 的预言**——减去 baseline 后，期望不动（末端回报差 < 0.3），只有方差在降（梯度估计范数显著变小）。

点题：策略梯度定理最漂亮的一步，是**环境模型 p(s′|s,a) 在求导中整个被消掉**——NB1/NB2 里精心搭的转移张量 T，在这一本里一次都不会出现。REINFORCE 对环境动力学一无所知，照样能学：它只需要"按策略采样 + 看奖励"。

## 怎么用这本笔记本

- **Shift + Enter** 逐格往下跑；带 `TODO` 的格子要**你自己补全代码**（签名、提示、形状都在注释里）；
- `✅ 自检` 格跑绿（不报错）才算过关；`🏔 挑战` 格没有答案，留给你的好奇心；
- 浏览器刷新会清空 kernel 状态——重进请从上往下重跑；
- 本本用到 **matplotlib** 画 10-seed 误差带：首次加载比 NB0 多下载约 10MB，之后走浏览器缓存；
- 最重的一格（10 seed × 2 模式 × 300 回合）在浏览器内核里约需 1 分钟量级，耐心等它跑完。

## 0 · 环境自检

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("numpy 版本    :", np.__version__)
print("matplotlib 版本:", plt.matplotlib.__version__)
assert int(np.__version__.split(".")[0]) in (1, 2), "numpy 主版本异常"
print("环境就绪：numpy + matplotlib（画 10-seed 误差带用）")

## 1 · 从定理到代码：两行数学

L9 用 20 步推出的策略梯度定理（定理 9.1），落到代码里只有两行数学：

$$\nabla_\theta J(\theta)=\mathbb{E}_{\tau\sim\pi_\theta}\Big[\sum_{t}\nabla_\theta\ln\pi(a_t\mid s_t,\theta)\,G_t\Big]$$

$$\nabla_\theta\ln\pi(a\mid s,\theta)=\mathrm{onehot}(a)-\pi(\cdot\mid s)\qquad(\text{softmax 的魔法导数})$$

- 第一行说：梯度是**一个期望**——期望就能用采样估计。一条回合算出的 $g=\sum_t\nabla\ln\pi(a_t|s_t)\,G_t$ 就是它的无偏估计："好经历加概率、坏经历减概率"。
- 第二行说：期望里的每一项都有闭式——被选动作的概率往上推、其余往下压，推力合计为零（**概率守恒**）。
- 两行合起来：**不需要状态分布、不需要转移概率**——log-derivative 把 ∇P(τ;θ) 里的环境模型整个吸掉了。这就是开头说的"T 张量不出现"。

先把两个零件备好。

In [ ]:
def softmax(h):
    """π(·|s) = softmax(θ[s-1])：分数 → 概率（减最大值，数值稳定）"""
    e = np.exp(h - h.max())
    return e / e.sum()

p = softmax(np.zeros(5))
assert np.allclose(p, 0.2), "全零分数 → 均匀策略"
q = softmax(np.array([1000.0, 1000.0, 999.0]))          # 分数再大也不上溢
assert np.isfinite(q).all() and abs(q.sum() - 1.0) < 1e-12
assert np.allclose(softmax(np.array([1.0, 2.0])), softmax(np.array([11.0, 12.0])))  # 平移不变
print("softmax 就绪：归一、稳定、平移不变 —— 概率永远现场归一化产出")

In [ ]:
def grad_ln_pi(pi_s, a):
    """∇θ ln π(a|s) = onehot(a) − π(·|s) —— 全文件最有含金量的一行"""
    g = -pi_s.copy()
    g[a] += 1.0
    return g

g = grad_ln_pi(np.full(5, 0.2), 1)
assert np.allclose(g, [-0.2, 0.8, -0.2, -0.2, -0.2])
assert abs(g.sum()) < 1e-12, "推力合计为零 —— 概率守恒（Σ_a ∇π = 0）"
print("∇lnπ 就绪：把被选动作的概率往上推、其余往下压，合计为零")

## 2 · 环境：内嵌 4×4 GridWorld（NB0 同款）

自包含复刻 NB0 的作业世界：4×4、起点 s1、目标 s12（+1）、禁区 s8/s10（撞上弹回并挨罚 −1）、出界 −1、γ=0.9。动作 5 个：**下 右 上 左 原**（下标 0..4）。

与 NB0 的唯一差别：`step` 直接吃**动作下标**——策略梯度代码里动作是 `rng.choice` 采出来的下标，省一层换算。

转移语义（作业代码规则，分支优先级 **出界 > 目标 > 禁区 > 普通**）：

| 动作结果 | 下一状态 | 奖励 |
|---|---|---|
| 出界 | 原地 | −1 |
| 进目标 s12 | s12，回合结束 | +1 |
| 撞禁区 s8/s10 | 弹回原地 | −1 |
| 普通 / 原地不动 | 正常移动 | 0 |

In [ ]:
SIZE = 4
NUM_STATES = SIZE * SIZE          # s1 .. s16
START, TARGET = 1, 12             # 起点 s1，目标 s12
FORBIDDEN = {8, 10}               # 禁区 s8, s10
GAMMA = 0.9
ACTION_SPACE = [(0, 1), (1, 0), (0, -1), (-1, 0), (0, 0)]   # 下 右 上 左 原

def s2xy(s):
    """状态编号 → (x, y)：s1=(0,0) s8=(3,1) s10=(1,2) s12=(3,2)"""
    i = int(s) - 1
    return i % SIZE, i // SIZE

def xy2s(x, y):
    return int(y) * SIZE + int(x) + 1

assert (NUM_STATES, START, TARGET, GAMMA) == (16, 1, 12, 0.9)
assert FORBIDDEN == {8, 10} and len(ACTION_SPACE) == 5
assert s2xy(12) == (3, 2) and all(xy2s(*s2xy(s)) == s for s in range(1, 17))
print("世界规格：4×4 / 起点 s1 / 禁区 s8,s10 / 目标 s12 / γ=0.9 / 动作 5 个")

In [ ]:
class GridWorld:
    """NB0 同款 4×4 网格世界（自包含瘦身版）。step 直接吃动作下标 0..4。

    语义优先级：出界（−1，原地）> 进目标（+1，结束）> 撞禁区（−1，弹回）> 普通（0）。
    """

    def __init__(self):
        self.agent_state = START

    def reset(self):
        self.agent_state = START
        return self.agent_state

    def _get_next_state_and_reward(self, state, action):
        x, y = s2xy(state)
        nx, ny = x + action[0], y + action[1]
        if not (0 <= nx < SIZE and 0 <= ny < SIZE):        # 1) 出界
            return state, -1.0
        ns = xy2s(nx, ny)
        if ns == TARGET:                                   # 2) 目标
            return ns, 1.0
        if ns in FORBIDDEN:                                # 3) 禁区：弹回
            return state, -1.0
        return ns, 0.0                                     # 4) 普通 / 原地

    def step(self, a):
        ns, r = self._get_next_state_and_reward(self.agent_state, ACTION_SPACE[a])
        done = ns == TARGET
        self.agent_state = ns
        return ns, r, done, {}

env = GridWorld()
env.reset()
assert env.step(2)[:3] == (1, -1.0, False)        # s1 向上：出界 −1，留在原地
env.agent_state = 11
assert env.step(1)[:3] == (12, 1.0, True)         # s11 右移进 s12：+1，done
env.agent_state = 7
assert env.step(1)[:3] == (7, -1.0, False)        # s7 右移撞 s8 禁区：−1，弹回
print("环境就绪：出界 / 目标 / 禁区三条转移规则与 NB0 逐条一致")

### 2.1 · 策略参数化：θ 是一张 16×5 的分数表

表格型 softmax 策略：每个状态配 5 个**原始分数**，参数就是一张 $\theta\in\mathbb{R}^{16\times5}$ 的表——

$$\pi(a\mid s,\theta)=\frac{\exp\,\theta[s\!-\!1,a]}{\sum_{a'}\exp\,\theta[s\!-\!1,a']}$$

两条纪律（L9 的「softmax 不归一化」误区）：

- θ **只存分数，不存概率**——概率永远由 softmax 现场归一化产出，更新只动分数；
- 分数差不能拉太开——softmax 退化成硬 argmax 时 ∇lnπ = onehot − π ≈ 0，策略"冻死"、探索失声。

In [ ]:
theta0 = np.zeros((NUM_STATES, 5))    # 参数矩阵：16 状态 × 5 动作，θ[s-1] ∈ R^5
pi0 = np.array([softmax(theta0[s - 1]) for s in range(1, NUM_STATES + 1)])

assert theta0.shape == (16, 5)
assert np.allclose(pi0, 0.2), "全零分数 → 每个状态都是均匀策略"
print("θ 的形状 (16, 5)：每行是一个状态的 5 个动作分数")
print("全零 θ → 均匀策略，前 3 个状态的 π(·|s)：")
print(pi0[:3].round(3))

## 3 · TODO 1 · 采样一条回合，并倒推 $G_t$

REINFORCE 是**蒙特卡洛**节奏：先按当前策略 π_θ 完整采一条回合，再从轨迹末端倒推每一步的折扣回报，最后一次性更新。

$$G_t = r_{t+1}+\gamma\,G_{t+1},\qquad G_{T-1}=r_T$$

**任务**：补全 `sample_episode`——

- 按当前策略采动作：`pi_s = softmax(theta[s-1])`，`a = int(rng.choice(5, p=pi_s))`；
- **所有**随机性都走 `rng`（后面要靠它做 seed 级复现）；
- `done=True` 立即结束；`max_steps` 是截断保险丝；
- 倒推：`acc = 0.0`，从 `t = T-1` 扫到 `0`，每步 `acc = gamma*acc + rewards[t]`。

In [ ]:
# ═════════ TODO 1 · 采样一条回合，并倒推 G_t ═════════
def sample_episode(env, theta, rng, max_steps=100, gamma=GAMMA):
    """按当前策略 π_θ = softmax(θ[s-1]) 采一整条回合。

    返回 (states, actions, rewards, G)：
      states[t] = s_t，actions[t] = a_t，rewards[t] = r_{t+1}
      G[t] = 从 t 起的折扣回报（G_t = r_{t+1} + γ·G_{t+1}，倒推）

    提示：
      · 采动作：a = int(rng.choice(5, p=softmax(theta[s-1])))
      · env.step(a) 直接吃动作下标；done=True 立即 break
      · 倒推：acc = 0.0；for t in reversed(range(T)): acc = gamma*acc + rewards[t]
    """
    s = env.reset()
    states, actions, rewards = [], [], []
    # TODO: 采样循环——max_steps 步内按 π 采动作 → step → 收集三元组 → done 即 break
    # TODO: 倒推 G——长度 T 的 numpy 数组
    # return np.array(states), np.array(actions), np.array(rewards), G
    raise NotImplementedError("TODO 1：补全采样循环与 G 的倒推")

In [ ]:
rng = np.random.default_rng(20260913)
states, actions, rewards, G = sample_episode(env, theta0, rng, max_steps=100)

# 可复现：同 seed、新环境重采一遍，轨迹必须逐位相同（随机性全在 rng 里）
states2, actions2, _, G2 = sample_episode(GridWorld(), theta0, np.random.default_rng(20260913), 100)
assert np.array_equal(states, states2) and np.array_equal(actions, actions2)

# 结构：四元组等长；G 满足递推式 G_t = r_{t+1} + γ·G_{t+1}
assert len(G) == len(states) == len(actions) == len(rewards)
assert np.allclose(G[-1], rewards[-1])
assert np.allclose(G[:-1], rewards[:-1] + GAMMA * G[1:])

# 合法性：状态与动作都在定义域内
assert all(1 <= s <= 16 for s in states) and all(0 <= a <= 4 for a in actions)

print(f"均匀策略下的一回合：长度 T={len(states)}  总奖励={rewards.sum():+.0f}  G_0={G[0]:+.3f}")
print("前 12 步  s:", states[:12])
print("         a:", actions[:12])
print("✅ TODO 1 通过：采样可复现，G 的倒推与递推式一致")

## 4 · TODO 2 · 梯度估计：$g=\sum_t c_t\,\nabla\ln\pi(a_t|s_t)$

一条回合的梯度估计。关键在**形状**：θ 是 16×5 的表，所以 $\nabla_\theta\ln\pi(a_t|s_t)$ 不是只有一个 5 维向量，而是一张 **(16, 5)** 的稀疏矩阵——**只有第 $s_t$ 行非零**，那一行 = $\mathrm{onehot}(a_t)-\pi(\cdot|s_t)$。同一状态在一回合里到访多次，贡献往同一行**累加**。

评分员 $c_t$ 先用 $G_t$ 本身（无 baseline）；`baseline` 参数留个口子——TODO 3 里传运行均值 $b$ 进来，$c_t$ 就换成 $G_t-b$（评分员可插拔：任何能打分的量都能往里塞）。

In [ ]:
# ═════════ TODO 2 · 梯度估计 g = Σ_t c_t · ∇lnπ(a_t|s_t) ═════════
def episode_gradient(theta, states, actions, G, baseline=0.0):
    """一条回合的 REINFORCE 梯度估计。

    返回 g，shape (16, 5)：
      每个时间步贡献 c_t · (onehot(a_t) − π(·|s_t))，只落在第 s_t−1 行
      评分员 c_t = G[t] − baseline（baseline=0 就是无 baseline 版）

    提示：g = np.zeros((NUM_STATES, 5)) 起手；逐时间步往 g[s-1] 行里累加；
          一行内的向量用 grad_ln_pi(softmax(theta[s-1]), a) 现算。
    """
    # TODO: 逐时间步累加 c_t * grad_ln_pi(softmax(theta[s-1]), a)
    # return g   # shape (16, 5)
    raise NotImplementedError("TODO 2：补全梯度估计（注意形状 16×5）")

In [ ]:
# 手算案例：s11 出发、向右一步进目标（a=1 是"右"），单步回合
#   π(·|s11) = 均匀 0.2 → ∇lnπ = onehot(1) − 0.2；G_0 = +1.0（首步无折扣）
g_hand = np.zeros((16, 5))
g_hand[11 - 1] = np.array([-0.2, 0.8, -0.2, -0.2, -0.2])     # ← 期望输出

g_test = episode_gradient(theta0, np.array([11]), np.array([1]), np.array([1.0]))
assert g_test.shape == (16, 5)
assert np.allclose(g_test, g_hand), "和手算对不上：检查是不是 (onehot(a) − π(·|s)) × G_t"

mask = np.ones(NUM_STATES, dtype=bool); mask[11 - 1] = False
assert np.allclose(g_test[mask], 0.0), "没到过的状态行应全零"
assert abs(g_test.sum()) < 1e-12, "全表元素和 ≈ 0 —— 概率守恒"

# baseline 口子：c_t = G_t − b；b 取 G_0 时本回合估计为零
g_b = episode_gradient(theta0, np.array([11]), np.array([1]), np.array([1.0]), baseline=1.0)
assert np.allclose(g_b, 0.0)
print("✅ TODO 2 通过：梯度估计与手算一致；只有到过的行非零；baseline 口子就位")

## 5 · TODO 3 · 训练循环：`train(seed, use_baseline, episodes)`

REINFORCE 更新只有一行：$\theta\leftarrow\theta+\alpha\,g$。

**baseline 分支**：`use_baseline=True` 时，评分员从 $G_t$ 换成 $G_t-b$，$b$ 是**回合回报的运行均值**（简化版 baseline）。注意顺序：先用**旧的** b 给本回合打分，回合结束后再让 b 吸收本回合的 $G_0$——不然估计量引用了自己，复现和推导都对不上。

> 这是**简化版**：书内的最优 baseline 是 $b=v_\pi(s)$——**逐状态**。全局运行均值只做掉"整体水平"的中心化；逐状态版本还能把"不同状态回报水平的差异"也中心化掉，方差再降一档。学一个 v 的同时拿 TD 误差 δ 当优势，就是第 10 章的 A2C——本本 §8 再回来对照。

默认超参数给好了：α=0.25、γ=0.9、max_steps=100。要记录两样东西：每回合的 $G_0$（`returns`）和梯度估计范数 ‖g‖（`g_norms`，§7 方差对比的主角）。

In [ ]:
# ═════════ TODO 3 · 训练循环 train(seed, use_baseline, episodes) ═════════
def train(seed, use_baseline=False, episodes=300, alpha=0.25, gamma=GAMMA, max_steps=100):
    """REINFORCE 训练：θ ← θ + α·g。

    use_baseline=False：评分员 c_t = G_t
    use_baseline=True ：评分员 c_t = G_t − b；b 是回合回报 G_0 的运行均值
                        （先用旧 b 打分，本回合结束后再更新 b）

    返回 dict(theta=θ, returns=每回合 G_0, g_norms=每回合 ‖g‖)，都是数组。
    """
    rng = np.random.default_rng(seed)
    env = GridWorld()
    theta = np.zeros((NUM_STATES, 5))
    b, n_seen = 0.0, 0                      # baseline 及其计数
    returns, g_norms = [], []
    # TODO: 每回合——sample_episode(...) → episode_gradient(..., baseline=b if use_baseline else 0.0)
    # TODO:           → θ ← θ + α·g → 更新运行均值 b → 记录 G_0 与 ‖g‖（np.linalg.norm）
    # return dict(theta=theta, returns=np.array(returns), g_norms=np.array(g_norms))
    raise NotImplementedError("TODO 3：补全训练循环")

In [ ]:
# 冒烟：30 回合，两种模式都应能跑通且逐位可复现
smoke_a = train(seed=0, use_baseline=False, episodes=30)
smoke_b = train(seed=0, use_baseline=True, episodes=30)

for sm in (smoke_a, smoke_b):
    assert len(sm["returns"]) == 30 and len(sm["g_norms"]) == 30
    assert np.all(np.isfinite(sm["returns"])) and np.all(np.isfinite(sm["g_norms"]))
    assert sm["theta"].shape == (16, 5) and np.isfinite(sm["theta"]).all()

smoke_a2 = train(seed=0, use_baseline=False, episodes=30)
assert np.array_equal(smoke_a["returns"], smoke_a2["returns"]), "同 seed 必须逐位可复现"
print(f"冒烟 30 回合：无 baseline 末端 G0 均值 {smoke_a['returns'][-10:].mean():+.3f}，"
      f"有 baseline {smoke_b['returns'][-10:].mean():+.3f}（还没学出来，正常）")
print("✅ TODO 3 通过：训练循环可复现，returns / g_norms / theta 记录齐全")

In [ ]:
def greedy_return(theta, start=START, max_steps=60):
    """greedy（argmax 分数）策略从 start 出发走一趟：返回未折扣总奖励。

    评估口径说明：干净走到 s12 = +1.0；折扣口径的上限只有 0.9^4 ≈ 0.656，
    为了阈值好读，评估一律用未折扣总奖励。
    """
    env_eval = GridWorld()
    env_eval.agent_state = start
    total, s = 0.0, start
    for _ in range(max_steps):
        a = int(np.argmax(theta[s - 1]))
        s, r, done, _ = env_eval.step(a)
        total += r
        if done:
            break
    return total

assert greedy_return(theta0) < 0, "全零分数的 greedy 一路向下撞墙，总奖励应为负"
print("未训练策略的 greedy 评估回报 =", greedy_return(theta0), "（argmax 全选'下'，一路撞南墙）")
print("评估函数就绪：最优 = +1.0（干净走到 s12）")

## 6 · 主实验：300 回合，无 baseline vs 有 baseline

推导 #11 的两条预言，先立字据：

1. **都能学到接近最优的策略**：从 s1 的 greedy 评估回报 ≥ 0.7（阈值实测留了裕度：干净走到 s12 = 1.0，撞一次墙就掉到 0 以下）；
2. **期望不变 → 殊途同归**：两种模式的末端回报差 < 0.3——减 baseline 不改变"要去的方向"。

In [ ]:
EPISODES = 300
res_plain = train(seed=0, use_baseline=False, episodes=EPISODES)
res_base = train(seed=0, use_baseline=True, episodes=EPISODES)

g_plain = greedy_return(res_plain["theta"])
g_base = greedy_return(res_base["theta"])
tail_plain = res_plain["returns"][-50:].mean()
tail_base = res_base["returns"][-50:].mean()

print(f"无 baseline：末端 50 回合平均 G0 = {tail_plain:+.3f}   greedy 评估回报 = {g_plain:+.2f}")
print(f"有 baseline：末端 50 回合平均 G0 = {tail_base:+.3f}   greedy 评估回报 = {g_base:+.2f}")
print(f"（参考：最优折扣 G0 = 0.9^4 = {GAMMA ** 4:.4f}；greedy 干净一趟 = +1.0）")

# ✅ 检查 1：两种模式都学到接近最优的策略
assert g_plain >= 0.7 and g_base >= 0.7, "300 回合应足够学出干净走到 s12 的 greedy 策略"
print("✅ 检查 1 通过：两种模式 greedy 评估回报均 ≥ 0.7 —— REINFORCE 学到了接近最优的策略")

# ✅ 检查 2：期望不变 —— 推导 #11 预言两种模式殊途同归
assert abs(g_base - g_plain) < 0.3, "末端回报差应 < 0.3：减 baseline 不改期望"
print("✅ 检查 2 通过：两种模式末端回报差 < 0.3 —— 期望没动（方向不变）")

In [ ]:
ACTION_NAMES = ["下", "右", "上", "左", "原"]

def show_policy(theta, title):
    """把 greedy 策略画成 4×4 字符网格（◎ 目标 / ✕ 禁区）"""
    print(title)
    for y in range(SIZE):
        row = []
        for x in range(SIZE):
            s = xy2s(x, y)
            if s == TARGET:
                row.append(" ◎")
            elif s in FORBIDDEN:
                row.append(" ✕")
            else:
                row.append(f" {ACTION_NAMES[int(np.argmax(theta[s - 1]))]}")
        print("".join(row))
    print()

show_policy(res_plain["theta"], "无 baseline 学到的 greedy 策略：")
show_policy(res_base["theta"], "有 baseline 学到的 greedy 策略：")
print("（s1 → s2 → s3 → s7 → s11 → s12 是 5 步最优路：右 右 下 下 右）")

## 7 · 方差对比：10 seed × 300 回合

单次训练的曲线一半是信号、一半是运气——要看清"方差"这件事，得**多 seed**。两种模式各跑 10 个 seed，然后看两张图：

- **左图（回报曲线，均值 + 误差带）**：两组的**均值应当相当**（期望不变——推导 #11 的前半句）；误差带更窄的那组更稳；
- **右图（每回合梯度估计范数 ‖g‖）**：在期望相同的前提下，噪声大 → 范数大。这是"方差变小"最直接的读数，也是下一个自检格的判据。

这一格是全本最重的一格（20 次训练），稍等片刻。

In [ ]:
SEEDS = list(range(10))
print(f"跑 {len(SEEDS)} 个 seed × 2 种模式 × {EPISODES} 回合……（全本最重的一格）")
runs_plain = [train(seed=s, use_baseline=False, episodes=EPISODES) for s in SEEDS]
runs_base = [train(seed=s, use_baseline=True, episodes=EPISODES) for s in SEEDS]

tail_p = np.mean([r["returns"][-50:].mean() for r in runs_plain])
tail_b = np.mean([r["returns"][-50:].mean() for r in runs_base])
greedy_p = np.mean([greedy_return(r["theta"]) for r in runs_plain])
greedy_b = np.mean([greedy_return(r["theta"]) for r in runs_base])
gsq_p = np.mean([(r["g_norms"] ** 2).mean() for r in runs_plain])
gsq_b = np.mean([(r["g_norms"] ** 2).mean() for r in runs_base])

print(f"末端 50 回合平均 G0（10 seed 均值）：无 baseline {tail_p:+.3f} | 有 baseline {tail_b:+.3f}")
print(f"greedy 评估回报   （10 seed 均值）：无 baseline {greedy_p:+.2f} | 有 baseline {greedy_b:+.2f}")
print(f"平均每回合梯度范数平方 ‖g‖²      ：无 baseline {gsq_p:.4f} | 有 baseline {gsq_b:.4f}")

In [ ]:
def rolling_mean(x, w=20):
    """窗口 w 的滑动平均（返回长度 len(x)-w+1 的曲线）"""
    return np.convolve(x, np.ones(w) / w, mode="valid")

Rp = np.array([rolling_mean(r["returns"]) for r in runs_plain])   # (10, 281)
Rb = np.array([rolling_mean(r["returns"]) for r in runs_base])
Gp = np.array([rolling_mean(r["g_norms"]) for r in runs_plain])
Gb = np.array([rolling_mean(r["g_norms"]) for r in runs_base])
x = np.arange(Rp.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(11, 4), dpi=110)

ax = axes[0]   # 左图：回报曲线 均值 + 误差带
ax.plot(x, Rp.mean(0), color="#c0392b", label="no baseline")
ax.fill_between(x, Rp.mean(0) - Rp.std(0), Rp.mean(0) + Rp.std(0), color="#c0392b", alpha=0.15)
ax.plot(x, Rb.mean(0), color="#2471a3", label="+ baseline (running mean)")
ax.fill_between(x, Rb.mean(0) - Rb.std(0), Rb.mean(0) + Rb.std(0), color="#2471a3", alpha=0.15)
ax.set(xlabel="episode (moving avg, w=20)", ylabel="discounted return G0",
       title="10-seed mean +/- std")
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]   # 右图：梯度估计范数（更新噪声）
ax.plot(x, Gp.mean(0), color="#c0392b", label="no baseline")
ax.plot(x, Gb.mean(0), color="#2471a3", label="+ baseline")
ax.set(xlabel="episode (moving avg, w=20)", ylabel="||g|| per-episode grad estimate",
       title="update noise")
ax.legend()
ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()
print("左：两组回报曲线均值相当、有 baseline 的带更窄（期望不变，更稳）；")
print("右：有 baseline 的梯度估计范数明显更小（同样的方向，更小的噪声）")

In [ ]:
ratio = gsq_b / gsq_p
print(f"平均 ‖g‖² 比值（有 baseline / 无 baseline）= {ratio:.3f}")
assert ratio < 1.0, "推导 #11 的预言：baseline 应显著降低梯度估计的噪声"
print(f"末端回报均值差（10 seed）：{abs(tail_b - tail_p):.3f}（再次印证期望没动）")
print("✅ 方差对比通过：baseline 组梯度估计范数显著更小 —— 方差降了，方向没变")

## 8 · 这就是推导 #11：「方向不变、噪声变小」

L10 那条推导链证的是：

$$\mathbb{E}\big[(G_t-b)\,\nabla\ln\pi\big]=\mathbb{E}\big[G_t\,\nabla\ln\pi\big]$$

因为 $b$ 不依赖动作：$\mathbb{E}[b\,\nabla\ln\pi]=b\sum_a\nabla\pi(a|s)=0$——**概率守恒**（概率和恒为 1，导数为 0）。所以减任何"不看动作"的量都**白赚**：期望不动，只剩降噪。对应刚才两个自检格：

| 推导 #11 的预言 | 实验读数 |
|---|---|
| 期望不变 | 末端回报差 < 0.3，10 seed 均值差同样很小 |
| 方差变小 | 平均 ‖g‖² 比值显著 < 1（你跑出的数就在上面一行） |

最后一段诚实声明：我们的 $b$ 是**全局运行均值**，比书内最优 $b=v_\pi(s)$ 粗糙——但"白赚"的方向已经看得见。逐状态基线（乃至直接拿 TD 误差 $\delta=r+\gamma v(s')-v(s)$ 当优势）能把方差再降一档：**演员上台表演、评论家台下打分**，那就是 actor-critic 的故事，也是这本实验室之外的下一段路。

## 🏔 挑战 · γ = 0.5 vs γ = 0.99

无答案，自己跑。把 `train(..., gamma=...)` 的 γ 换成 0.5 和 0.99 各跑几个 seed：

- **方差**：γ 越大，$G_t$ 的贡献链越长——一条轨迹的运气乘得越深，平均 ‖g‖² 会怎么变？
- **收敛速度**：γ 小评分短视，是不是更快？γ=0.5 时 4 步外的 +1 只值 $0.5^4\approx0.06$——策略还会在乎 s12 吗？会不会先学会"少挨罚"而放弃远处的 +1？
- 用与 §7 同款的两张图对比三种 γ，写下你的结论。

**附加题**：把 baseline 从"全局运行均值"升级成"**逐状态**运行均值 b(s)"（向 $v_\pi(s)$ 靠拢）——平均 ‖g‖² 还能再降多少？

In [ ]:
# 🏔 挑战格（无答案）：γ = 0.5 vs 0.99 —— 方差与收敛速度怎么变？
#
# 骨架（自己补全）：
#   for g in (0.5, 0.99):
#       runs = [train(seed=s, use_baseline=False, episodes=300, gamma=g) for s in range(5)]
#       ...  记录末端均值与平均 ‖g‖²，和 γ=0.9 的基准对比 ...
#   ...  用 §7 同款的 rolling_mean 画图 ...
#
# 附加题骨架：
#   把 train 里 baseline 从标量 b 换成长度 16 的数组 b_vec（逐状态运行均值）
#   —— 注意 episode_gradient 的 baseline 参数也要跟着向量化。

print("🏔 挑战格：填上你自己的实验代码，跑出 γ 与逐状态 baseline 的结论")

## 9 · 收官：NB0 → NB6，一条谱系

| 本 | 方法 | 要环境模型吗 | 学什么 |
|---|---|---|---|
| NB1/NB2 | 值迭代 / 策略迭代 | **要**（整张 T、R） | 价值 |
| NB4 | Q-learning | 不要 | 动作价值 |
| NB5 | TD-Linear | 不要 | 价值的线性近似 |
| **NB6** | **REINFORCE** | **不要** | **策略本身** |

从 DP 的"模型已知、精确求解"，到 MC/TD 的"模型未知、采样估计"，再到策略梯度的"模型被推导整个消掉"——六个笔记本走完了强化学习方法论的这条主线。你在这一本亲手验证了三件事：

1. 策略梯度定理落成一行代码：∇lnπ = onehot(a) − π；
2. **期望不变**：减 baseline 后两种模式末端回报差 < 0.3；
3. **方差变小**：平均 ‖g‖² 比值显著 < 1。

## 🎓 跑通即毕业 · 下一步：真网络

三个 ✅ 自检格全绿 = NB6 毕业 = 笔记本实验室全部通关。

**下一步：真网络。** JupyterLite 的浏览器内核装不下 PyTorch——想用真网络跑 DQN / Actor-Critic，**上传本文件到 Google Colab** 即可继续（免费 GPU，本本的 numpy 代码在 Colab 里原样能跑）：

> 站外链接：[colab.research.google.com](https://colab.research.google.com/) · 上传方式：File → Upload notebook（Colab 是站外服务，可达性自行评估）